# 4. Ensembles de Arboles de Decision

Un arbol de decisión es un modelo débil, el aumento del poder predictivo proviene al ensamblar varios arboles de decisión.
<br> Si promedio n arboles identicos, el resultados es exactamente el mismo que utilizar un solo arbol, necesito PERTURBAR cada arbol para disponer de variablidad

la variabilidad provendrá de estas fuentes:


*   Perturbar el dataset
*   Perturbar el algoritmo del arbol
*   Perturbar el dataset y el algoritmo del arbol al mismo tiempo

Se verán estos tres algoritmos


*   Arboles Azarosos
*   Random Forest
*   Gradient Boosting of Decision Trees

#### 4.01 Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Drive already mounted at /content/.drive; to attempt to forcibly remount, call drive.mount("/content/.drive", force_remount=True).


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"



---



## 4.02 Arboles Azarosos

Arboles Azarosos es el nombre de un algoritmo trivial (por favor NO confundir con Random Forest)
Qué tipo de perturbaciones se realizan en Arboles Azarosos
* Se perturba el dataset
* No se perturba el algoritmo, es siempre rpart original

Cada  arbolito de  Arboles Azarosos se entrena sobre un dataset perturbado,  que tiene exactamente la misma cantidad de registros pero solo un subconjunto de los atributos (campos)  del dataset, tomados al azar, de los originales.
<br> En esta primera corrida, se construira cada arbol en un dataset utilizando el 50% de los campos

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Mon Aug 24 04:12:55 AM 2026"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671287,35.9,1473300,78.7,1473300,78.7
Vcells,1242646,9.5,8388608,64.0,1978712,15.1


In [3]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: rpart



Aqui debe cargar SU semilla primigenia

In [4]:
"""
PARAM <- list()
PARAM$semilla_primigenia <- 378089

# parametros  arbol
# entreno cada arbol con solo 50% de las variables variables
#  por ahora, es fijo
PARAM$feature_fraction <- 0.5

PARAM$rpart$cp <- -1
PARAM$rpart$minsplit <- 500
PARAM$rpart$minbucket <- 180
PARAM$rpart$maxdepth <- 7

# voy a generar 512 arboles,
#  a mas arboles mas tiempo de proceso y MEJOR MODELO,
#  pero ganancias marginales
PARAM$num_trees_max <- 512
"""

ERROR: Error in parse(text = input): <text>:1:3: unexpected string constant
18: PARAM$num_trees_max <- 512
19: "
      ^


In [20]:
# nueva celda

PARAM <- list()
PARAM$semilla_primigenia <- 378089

# PAra identificar corrida con nueva grilla
PARAM$tag_experimento <- "colab_v2"
PARAM$competencia <- "utn-2026-inicial"

# ahora son rangos a explorar, no valores fijos
PARAM$rango_feature_fraction <- c(0.5 )
PARAM$rango_minsplit         <- c(200,400, 500, 600, 700)
PARAM$rango_minbucket        <- c(50,100, 150, 175)
PARAM$rango_maxdepth         <- c(10, 12, 14)

PARAM$cp_fijo <- -1   # se mantiene fijo, no se itera

# voy a generar 32 arboles por cada combinacion de hiperparametros
PARAM$num_trees_max <- 32

In [21]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp420_32_g2"     # <<< carpeta nueva
dir.create(experimento, showWarnings = FALSE)
setwd(paste0("/content/buckets/b1/exp/", experimento))

PARAM$tag_experimento <- "colab_v2"   # el tag sigue siendo importante para el analisis despues

In [7]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [8]:
# defino los dataset de entrenamiento y aplicacion
dtrain <- dataset[foto_mes == 202107]
dfuture <- dataset[foto_mes == 202109]

# arreglo clase_ternaria por algun distraido ""
dfuture[, clase_ternaria := NA ]

In [9]:
# Establezco cuales son los campos que puedo usar para la prediccion
# el copy() es por la Lazy Evaluation
campos_buenos <- copy(setdiff(colnames(dtrain), c("clase_ternaria")))

In [10]:
# que tamanos de ensemble grabo a disco
# grabar <- c(1, 2, 4, 8, 16, 32, 64, 128, 256, 384, 512)

In [11]:
# tb_prediccion <- dfuture[, list(numero_de_cliente)]
# aqui se va acumulando la probabilidad del ensemble
# tb_prediccion[, prob_acumulada := 0]

In [12]:
# set.seed(PARAM$semilla_primigenia) # Establezco la semilla aleatoria

In [13]:
"""
for (arbolito in seq(PARAM$num_trees_max) ) {
  message( arbolito, " ")
  qty_campos_a_utilizar <- as.integer(length(campos_buenos)
    * PARAM$feature_fraction)

  # elijo los campos al azar
  campos_random <- sample(campos_buenos, qty_campos_a_utilizar)

  # paso de un vector a un string con los elementos
  # separados por un signo de "+"
  # este hace falta para la formula
  campos_random <- paste(campos_random, collapse= " + ")

  # armo la formula para rpart
  formulita <- paste0("clase_ternaria ~ ", campos_random)

  # genero el arbol de decision
  modelo <- rpart(formulita,
    data= dtrain,
    xval= 0,
    control= PARAM$rpart
  )

  # aplico el modelo a los datos que no tienen clase
  prediccion <- predict(modelo, dfuture, type= "prob")

  tb_prediccion[, prob_acumulada := prob_acumulada + prediccion[, "BAJA+2"]]

  if (arbolito %in% grabar) {
    umbral_corte <- (1 / 40) * arbolito
    tb_prediccion[, Predicted := as.numeric(prob_acumulada > umbral_corte)]

    archivo_kaggle <- paste0(
        "KA420_",
        sprintf("%.3d", arbolito), # para que tenga ceros adelante
        ".csv"
      )

    # grabo el archivo
    fwrite( tb_prediccion[, list(numero_de_cliente, Predicted)],
      file= archivo_kaggle,
      sep= ","
    )

    # subida a Kaggle
    comando <- "kaggle competitions submit"
    competencia <- "-c utn-2026-inicial"
    arch <- paste( "-f", archivo_kaggle)

    mensaje <- paste0("-m 'cp=", PARAM$rpart$cp, "  minsplit=", PARAM$rpart$minsplit, "  minbucket=", PARAM$rpart$minbucket, " maxdepth=", PARAM$rpart$maxdepth, "'" )
    linea <- paste( comando, competencia, arch, mensaje)
    salida <- system(linea, intern=TRUE)
    cat(salida)
  }
}
"""

ERROR: Error in parse(text = input): <text>:1:3: unexpected string constant
2: for (arbolito in seq(PARAM$num_trees_max) ) {
3:   message( arbolito, "
     ^


In [14]:
# armar la grilla y persistirla
archivo_grilla <- paste0("grilla_hiperparams_", PARAM$tag_experimento, ".RDS")  # <<< nombre distinto

if (file.exists(archivo_grilla)) {

  grilla_hiperparams <- readRDS(archivo_grilla)
  cat("Grilla existente cargada.",
      sum(grilla_hiperparams$estado == "pendiente"), "pendientes de",
      nrow(grilla_hiperparams), "totales.\n")

} else {

  grilla_hiperparams <- CJ(
    feature_fraction = PARAM$rango_feature_fraction,
    minsplit          = PARAM$rango_minsplit,
    minbucket         = PARAM$rango_minbucket,
    maxdepth          = PARAM$rango_maxdepth
  )

  grilla_hiperparams <- grilla_hiperparams[minsplit >= 2 * minbucket]

  grilla_hiperparams[, cp := PARAM$cp_fijo]
  grilla_hiperparams[, id := .I]
  grilla_hiperparams[, tag := PARAM$tag_experimento]
  grilla_hiperparams[, estado := "pendiente"]

  saveRDS(grilla_hiperparams, archivo_grilla)
  cat("Grilla nueva creada con", nrow(grilla_hiperparams), "combinaciones validas.\n")
}


Grilla existente cargada. 54 pendientes de 54 totales.


In [15]:
# función que entrena UN ensemble de 32 árboles
# para UNA combinación de hiperparámetros y sube a Kaggle

generar_y_subir <- function(feature_fraction, vcp, minsplit, minbucket, maxdepth, id_combo, tag) {

  tb_prediccion <- dfuture[, list(numero_de_cliente)]
  tb_prediccion[, prob_acumulada := 0]

  control_arbol <- list(cp = vcp, minsplit = minsplit, minbucket = minbucket, maxdepth = maxdepth)
  set.seed(PARAM$semilla_primigenia)

  for (arbolito in seq(PARAM$num_trees_max)) {
    qty_campos_a_utilizar <- as.integer(length(campos_buenos) * feature_fraction)
    campos_random <- sample(campos_buenos, qty_campos_a_utilizar)
    formulita <- paste0("clase_ternaria ~ ", paste(campos_random, collapse = " + "))

    modelo <- rpart(formulita, data = dtrain, xval = 0, control = control_arbol)
    prediccion <- predict(modelo, dfuture, type = "prob")
    tb_prediccion[, prob_acumulada := prob_acumulada + prediccion[, "BAJA+2"]]
  }

  tb_prediccion[, prob_promedio := prob_acumulada / PARAM$num_trees_max]
  tb_prediccion[, Predicted := as.numeric(prob_promedio > 0.025)]

  # <<< nombre de archivo incluye el tag
  archivo_kaggle <- paste0("KA420_", tag, "_combo", sprintf("%.4d", id_combo), ".csv")

  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)], file = archivo_kaggle, sep = ",")

  # <<< mensaje incluye el tag
  mensaje <- paste0("-m 'tag=", tag,
                     " id=", id_combo,
                     " ff=", feature_fraction,
                     " cp=", vcp,
                     " minsplit=", minsplit,
                     " minbucket=", minbucket,
                     " maxdepth=", maxdepth, "'")

  linea <- paste("kaggle competitions submit -c", PARAM$competencia,
                  "-f", archivo_kaggle, mensaje)
  salida <- system(linea, intern = TRUE)
  cat(salida, "\n")
}

In [16]:
# loop principal
# solo procesa lo "pendiente", guarda checkpoint tras cada submit

pendientes <- which(grilla_hiperparams$estado == "pendiente")

for (i in pendientes) {

  fila <- grilla_hiperparams[i]

  cat("\nCombinacion", fila$id, "(", fila$tag, ") / ", nrow(grilla_hiperparams),
      " | ff=", fila$feature_fraction, " minsplit=", fila$minsplit,
      " minbucket=", fila$minbucket, " maxdepth=", fila$maxdepth, "\n")

  resultado_ok <- tryCatch({
    generar_y_subir(
      feature_fraction = fila$feature_fraction,
      vcp              = fila$cp,
      minsplit         = fila$minsplit,
      minbucket        = fila$minbucket,
      maxdepth         = fila$maxdepth,
      id_combo         = fila$id,
      tag              = fila$tag
    )
    TRUE
  }, error = function(e) {
    cat("  ERROR:", conditionMessage(e), "\n")
    FALSE
  })

  if (resultado_ok) {
    grilla_hiperparams[id == fila$id, estado := "subido"]
    saveRDS(grilla_hiperparams, archivo_grilla)
  } else {
    cat("\n  Cortando el loop.\n")
    break
  }

  Sys.sleep(2)
}


Combinacion 1 ( colab_v2 ) /  54  | ff= 0.5  minsplit= 200  minbucket= 50  maxdepth= 10 


Warning message in system(linea, intern = TRUE):
“running command 'kaggle competitions submit -c  -f KA420_colab_v2_combo0001.csv -m 'tag=colab_v2 id=1 ff=0.5 cp=-1 minsplit=200 minbucket=50 maxdepth=10'' had status 2”


 

Combinacion 2 ( colab_v2 ) /  54  | ff= 0.5  minsplit= 200  minbucket= 50  maxdepth= 12 


Warning message in system(linea, intern = TRUE):
“running command 'kaggle competitions submit -c  -f KA420_colab_v2_combo0002.csv -m 'tag=colab_v2 id=2 ff=0.5 cp=-1 minsplit=200 minbucket=50 maxdepth=12'' had status 2”


 

Combinacion 3 ( colab_v2 ) /  54  | ff= 0.5  minsplit= 200  minbucket= 50  maxdepth= 14 


...

Warning message in system(linea, intern = TRUE):
“running command 'kaggle competitions submit -c  -f KA420_colab_v2_combo0009.csv -m 'tag=colab_v2 id=9 ff=0.5 cp=-1 minsplit=400 minbucket=50 maxdepth=14'' had status 2”


 

Combinacion 10 ( colab_v2 ) /  54  | ff= 0.5  minsplit= 400  minbucket= 100  maxdepth= 10 


Warning message in system(linea, intern = TRUE):
“running command 'kaggle competitions submit -c  -f KA420_colab_v2_combo0010.csv -m 'tag=colab_v2 id=10 ff=0.5 cp=-1 minsplit=400 minbucket=100 maxdepth=10'' had status 2”


 

Combinacion 11 ( colab_v2 ) /  54  | ff= 0.5  minsplit= 400  minbucket= 100  maxdepth= 12 


Warning message in system(linea, intern = TRUE):
“running command 'kaggle competitions submit -c  -f KA420_colab_v2_combo0011.csv -m 'tag=colab_v2 id=11 ff=0.5 cp=-1 minsplit=400 minbucket=100 maxdepth=12'' had status 2”


In [ ]:
# función que entrena UN ensemble de 32 árboles
# para UNA combinación de hiperparámetros y sube a Kaggle

generar_y_subir <- function(feature_fraction, vcp, minsplit, minbucket, maxdepth, id_combo, tag) {

  tb_prediccion <- dfuture[, list(numero_de_cliente)]
  tb_prediccion[, prob_acumulada := 0]

  control_arbol <- list(cp = vcp, minsplit = minsplit, minbucket = minbucket, maxdepth = maxdepth)
  set.seed(PARAM$semilla_primigenia)

  for (arbolito in seq(PARAM$num_trees_max)) {
    qty_campos_a_utilizar <- as.integer(length(campos_buenos) * feature_fraction)
    campos_random <- sample(campos_buenos, qty_campos_a_utilizar)
    formulita <- paste0("clase_ternaria ~ ", paste(campos_random, collapse = " + "))

    modelo <- rpart(formulita, data = dtrain, xval = 0, control = control_arbol)
    prediccion <- predict(modelo, dfuture, type = "prob")
    tb_prediccion[, prob_acumulada := prob_acumulada + prediccion[, "BAJA+2"]]
  }

  tb_prediccion[, prob_promedio := prob_acumulada / PARAM$num_trees_max]
  tb_prediccion[, Predicted := as.numeric(prob_promedio > 0.025)]

  archivo_kaggle <- paste0("KA420_", tag, "_combo", sprintf("%.4d", id_combo), ".csv")
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)], file = archivo_kaggle, sep = ",")

  mensaje <- paste0("-m 'tag=", tag,
                     " id=", id_combo,
                     " ff=", feature_fraction,
                     " cp=", vcp,
                     " minsplit=", minsplit,
                     " minbucket=", minbucket,
                     " maxdepth=", maxdepth, "'")

  linea <- paste("kaggle competitions submit -c", PARAM$competencia,
                  "-f", archivo_kaggle, mensaje)

  salida <- system(linea, intern = TRUE)
  status <- attr(salida, "status")
  cat(salida, "\n")

  # <<< NUEVO: si el comando fallo, lanzo un error real para que el tryCatch lo capture
  if (!is.null(status) && status != 0) {
    stop(paste("Fallo el submit a Kaggle (status", status, "):", paste(salida, collapse = " ")))
  }
}

In [ ]:
# función que entrena UN ensemble de 32 árboles
# para UNA combinación de hiperparámetros y sube a Kaggle

generar_y_subir <- function(feature_fraction, vcp, minsplit, minbucket, maxdepth, id_combo, tag) {

  tb_prediccion <- dfuture[, list(numero_de_cliente)]
  tb_prediccion[, prob_acumulada := 0]

  control_arbol <- list(cp = vcp, minsplit = minsplit, minbucket = minbucket, maxdepth = maxdepth)
  set.seed(PARAM$semilla_primigenia)

  for (arbolito in seq(PARAM$num_trees_max)) {
    qty_campos_a_utilizar <- as.integer(length(campos_buenos) * feature_fraction)
    campos_random <- sample(campos_buenos, qty_campos_a_utilizar)
    formulita <- paste0("clase_ternaria ~ ", paste(campos_random, collapse = " + "))

    modelo <- rpart(formulita, data = dtrain, xval = 0, control = control_arbol)
    prediccion <- predict(modelo, dfuture, type = "prob")
    tb_prediccion[, prob_acumulada := prob_acumulada + prediccion[, "BAJA+2"]]
  }

  tb_prediccion[, prob_promedio := prob_acumulada / PARAM$num_trees_max]
  tb_prediccion[, Predicted := as.numeric(prob_promedio > 0.025)]

  # <<< nombre de archivo incluye el tag
  archivo_kaggle <- paste0("KA420_", tag, "_combo", sprintf("%.4d", id_combo), ".csv")

  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)], file = archivo_kaggle, sep = ",")

  # <<< mensaje incluye el tag
  mensaje <- paste0("-m 'tag=", tag,
                     " id=", id_combo,
                     " ff=", feature_fraction,
                     " cp=", vcp,
                     " minsplit=", minsplit,
                     " minbucket=", minbucket,
                     " maxdepth=", maxdepth, "'")

  linea <- paste("kaggle competitions submit -c", PARAM$competencia,
                  "-f", archivo_kaggle, mensaje)
  salida <- system(linea, intern = TRUE)
  cat(salida, "\n")
}

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

# Subir los csv generados y no subidos

In [30]:
archivos_generados <- list.files(pattern = "KA420_colab_v2_combo[0-9]+\\.csv")
ids_con_csv <- as.integer(gsub(".*combo([0-9]+)\\.csv", "\\1", archivos_generados))

cat("CSVs ya generados en disco:", length(ids_con_csv), "\n")
print(sort(ids_con_csv))

pendientes_con_csv <- grilla_hiperparams[id %in% ids_con_csv & estado == "pendiente"]
cat("De esos, pendientes de subir:", nrow(pendientes_con_csv), "\n")

for (i in seq_len(nrow(pendientes_con_csv))) {
  fila <- pendientes_con_csv[i]

  archivo_kaggle <- paste0("KA420_", fila$tag, "_combo", sprintf("%.4d", fila$id), ".csv")

  mensaje <- paste0("-m 'tag=", fila$tag,
                     " id=", fila$id,
                     " ff=", fila$feature_fraction,
                     " cp=", fila$cp,
                     " minsplit=", fila$minsplit,
                     " minbucket=", fila$minbucket,
                     " maxdepth=", fila$maxdepth, "'")

  linea <- paste("kaggle competitions submit -c", PARAM$competencia,
                  "-f", archivo_kaggle, mensaje)

  salida <- system(linea, intern = TRUE)
  status <- attr(salida, "status")
  cat(salida, "\n")

  if (is.null(status) || status == 0) {
    grilla_hiperparams[id == fila$id, estado := "subido"]
    saveRDS(grilla_hiperparams, archivo_grilla)
    cat("  -> OK, id", fila$id, "subido\n")
  } else {
    cat("  -> FALLO id", fila$id, ", deteniendo\n")
    break
  }

  Sys.sleep(2)
}

table(grilla_hiperparams$estado)

CSVs ya generados en disco: 11 
 [1]  1  2  3  4  5  6  7  8  9 10 11
De esos, pendientes de subir: 11 
  -> OK, id 1 subido
  -> OK, id 2 subido
  -> OK, id 3 subido
  -> OK, id 4 subido
  -> OK, id 5 subido
  -> OK, id 6 subido
  -> OK, id 7 subido
  -> OK, id 8 subido
  -> OK, id 9 subido
  -> OK, id 10 subido
  -> OK, id 11 subido



pendiente    subido 
       43        11 

In [24]:
# reviso que dice la grilla actualmente
table(grilla_hiperparams$estado)
print(grilla_hiperparams[id %in% 1:15])


pendiente    subido 
       43        11 

Key: <feature_fraction, minsplit, minbucket, maxdepth>
    feature_fraction minsplit minbucket maxdepth    id      tag    estado    cp
               <num>    <num>     <num>    <num> <int>   <char>    <char> <num>
 1:              0.5      200        50       10     1 colab_v2    subido    -1
 2:              0.5      200        50       12     2 colab_v2    subido    -1
 3:              0.5      200        50       14     3 colab_v2    subido    -1
 4:              0.5      200       100       10     4 colab_v2    subido    -1
 5:              0.5      200       100       12     5 colab_v2    subido    -1
 6:              0.5      200       100       14     6 colab_v2    subido    -1
 7:              0.5      400        50       10     7 colab_v2    subido    -1
 8:              0.5      400        50       12     8 colab_v2    subido    -1
 9:              0.5      400        50       14     9 colab_v2    subido    -1
10:              0.5      400       100       10    10 colab_v2  

In [25]:
cat("PARAM$competencia: '", PARAM$competencia, "'\n", sep="")

PARAM$competencia: 'utn-2026-inicial'


In [26]:
raw <- system("kaggle competitions submissions -c utn-2026-inicial -v --page-size 200", intern = TRUE)
raw <- raw[!grepl("^Warning", raw)]
subs_check <- fread(paste(raw, collapse = "\n"), header = TRUE, fill = TRUE)

# busco si aparece ALGUNA combinacion de colab_v2 en el historial real
subs_check[grepl("colab_v2", fileName)]

fileName,date,description,status,publicScore,privateScore
<chr>,<dttm>,<chr>,<chr>,<dbl>,<chr>


In [27]:
# si el chequeo anterior confirmo que NO hay ninguna colab_v2 en Kaggle:
grilla_hiperparams[id %in% 1:11, estado := "pendiente"]
saveRDS(grilla_hiperparams, archivo_grilla)

table(grilla_hiperparams$estado)


pendiente 
       54 

In [29]:
grilla_hiperparams[id %in% 1:11, estado := "pendiente"]
saveRDS(grilla_hiperparams, archivo_grilla)

table(grilla_hiperparams$estado)


pendiente 
       54 

# Extraccion de datos de KAGGLE

In [ ]:
nombre_competencia <- "utn-2026-inicial"   # confirmá que este es el nombre exacto de tu competencia

submissions_raw <- system(
  paste0("kaggle competitions submissions -c ", nombre_competencia, " -v"),
  intern = TRUE
)

submissions <- fread(paste(submissions_raw, collapse = "\n"))
print(names(submissions))
head(submissions)

Warning message in fread(paste(submissions_raw, collapse = "\n")):
“Discarded single-line footer: <<KA420_320014.csv,2026-08-23 10:59:23.483000,id=14 ff=0.3 cp=-1 minsplit=500 minbucket=100 maxdepth=7,SubmissionStatus.COMPLETE,341.415,>>”


[1] "fileName"     "date"         "description"  "status"       "publicScore" 
[6] "privateScore"


fileName,date,description,status,publicScore,privateScore
<chr>,<dttm>,<chr>,<chr>,<dbl>,<chr>
KA420_combo0063.csv,2026-08-24 02:37:54,id=63 ff=0.5 cp=-1 minsplit=500 minbucket=100 maxdepth=8,SubmissionStatus.COMPLETE,357.081,
KA420_combo0062.csv,2026-08-24 02:23:38,id=62 ff=0.5 cp=-1 minsplit=500 minbucket=100 maxdepth=7,SubmissionStatus.COMPLETE,351.831,
KA420_320061.csv,2026-08-24 02:04:03,id=61 ff=0.5 cp=-1 minsplit=500 minbucket=100 maxdepth=6,SubmissionStatus.COMPLETE,351.415,
KA420_320060.csv,2026-08-24 01:41:43,id=60 ff=0.5 cp=-1 minsplit=400 minbucket=200 maxdepth=8,SubmissionStatus.COMPLETE,345.581,
KA420_320059.csv,2026-08-24 01:10:56,id=59 ff=0.5 cp=-1 minsplit=400 minbucket=200 maxdepth=7,SubmissionStatus.COMPLETE,342.581,
KA420_320058.csv,2026-08-24 00:44:30,id=58 ff=0.5 cp=-1 minsplit=400 minbucket=200 maxdepth=6,SubmissionStatus.COMPLETE,348.165,


In [ ]:
nrow(submissions)
print(submissions)

[1] 49

               fileName                date
                 <char>              <POSc>
 1: KA420_combo0063.csv 2026-08-24 02:37:54
 2: KA420_combo0062.csv 2026-08-24 02:23:38
 3:    KA420_320061.csv 2026-08-24 02:04:03
 4:    KA420_320060.csv 2026-08-24 01:41:43
 5:    KA420_320059.csv 2026-08-24 01:10:56
 6:    KA420_320058.csv 2026-08-24 00:44:30
 7:    KA420_320057.csv 2026-08-24 00:22:21
 8:    KA420_320056.csv 2026-08-23 23:53:00
 9:    KA420_320055.csv 2026-08-23 23:27:25
10:    KA420_320054.csv 2026-08-23 23:05:36
11:    KA420_320053.csv 2026-08-23 22:36:01
12:    KA420_320052.csv 2026-08-23 22:10:10
13:    KA420_320051.csv 2026-08-23 21:48:35
14:    KA420_320050.csv 2026-08-23 21:19:43
15:    KA420_320049.csv 2026-08-23 20:54:28
16:    KA420_320048.csv 2026-08-23 20:32:51
17:    KA420_320047.csv 2026-08-23 20:15:12
18:    KA420_320046.csv 2026-08-23 19:59:27
19:    KA420_320045.csv 2026-08-23 19:46:02
20:    KA420_320044.csv 2026-08-23 19:01:07
21:    KA420_320043.csv 2026-08-

In [ ]:
help_output <- system("kaggle competitions submissions --help", intern = TRUE)
cat(help_output, sep = "\n")

usage: kaggle competitions submissions [-h] [-v] [-q] [--page-size PAGE_SIZE]
                                       [--page-token PAGE_TOKEN]
                                       [competition]

options:
  -h, --help            show this help message and exit
  competition           Competition URL suffix (use "kaggle competitions list" to show options)
                        If empty, the default competition will be used (use "kaggle config set competition")"
  -v, --csv             Print results in CSV format (if not set print in table format)
  -q, --quiet           Suppress printing information about the upload/download progress
  --page-size PAGE_SIZE
                        Number of items to show on a page. Default size is 20, max is 200
  --page-token PAGE_TOKEN
                        Page token for results paging.


In [ ]:
submissions_raw <- system(
  paste0("kaggle competitions submissions -c ", nombre_competencia, " -v --page-size 200"),
  intern = TRUE
)

submissions <- fread(paste(submissions_raw, collapse = "\n"), header = TRUE, fill = TRUE)
nrow(submissions)

[1] 83

In [ ]:
submissions_raw <- system(
  paste0("kaggle competitions submissions -c ", nombre_competencia, " -v --page-size 200"),
  intern = TRUE
)

# saco cualquier linea que sea un warning, no datos reales
submissions_raw <- submissions_raw[!grepl("^Warning", submissions_raw)]

submissions <- fread(paste(submissions_raw, collapse = "\n"), header = TRUE, fill = TRUE)
print(names(submissions))
nrow(submissions)

[1] "fileName"     "date"         "description"  "status"       "publicScore" 
[6] "privateScore"


[1] 82

In [ ]:
submissions_experimento <- submissions[grepl("KA420_", fileName)]
cat("Total de submissions del experimento:", nrow(submissions_experimento), "\n")

submissions_experimento[, origen := ifelse(grepl("KA420_combo", fileName), "local", "colab")]
submissions_experimento[, id := as.integer(gsub(".*id=([0-9]+).*", "\\1", description))]
submissions_experimento[, publicScore := as.numeric(publicScore)]

submissions_validas <- submissions_experimento[!is.na(id)]
cat("Con id valido:", nrow(submissions_validas), "\n")
cat("IDs duplicados:", sum(duplicated(submissions_validas$id)), "\n")

Total de submissions del experimento: 75 


Warning message in eval(jsub, SDenv, parent.frame()):
“NAs introduced by coercion”


Con id valido: 64 
IDs duplicados: 0 


In [ ]:
if (sum(duplicated(submissions_validas$id)) > 0) {
  ids_repetidos <- submissions_validas[duplicated(id) | duplicated(id, fromLast = TRUE)][order(id)]
  print(ids_repetidos[, list(id, fileName, date, publicScore, status)])
}

In [ ]:
grilla_hiperparams <- readRDS("grilla_hiperparams.RDS")

resultados_kaggle <- merge(
  grilla_hiperparams,
  submissions_validas[, list(id, publicScore, status, date, origen)],
  by = "id",
  all.x = TRUE
)

resultados_kaggle <- resultados_kaggle[order(-publicScore)]
fwrite(resultados_kaggle, "resultados_kaggle_completo.csv")

print(resultados_kaggle[1:15])
cat("\nCon score valido:", sum(!is.na(resultados_kaggle$publicScore)),
    "de", nrow(resultados_kaggle), "combinaciones totales de la grilla\n")

       id feature_fraction minsplit minbucket maxdepth    cp    estado
    <int>            <num>    <num>     <num>    <num> <num>    <char>
 1:    63              0.5      500       100        8    -1 pendiente
 2:    50              0.5      400       100        7    -1    subido
 3:    51              0.5      400       100        8    -1    subido
 4:    49              0.5      400       100        6    -1    subido
 5:    62              0.5      500       100        7    -1 pendiente
 6:    61              0.5      500       100        6    -1    subido
 7:    64              0.5      500       150        6    -1 pendiente
 8:    45              0.3      700       175        8    -1    subido
 9:    58              0.5      400       200        6    -1    subido
10:    60              0.5      400       200        8    -1    subido
11:    55              0.5      400       175        6    -1    subido
12:    52              0.5      400       150        6    -1    subido
13:   



---

